In [1]:
import pandas as pd

# wczytuje oryginalny plik
df = pd.read_csv("open_library_articles.csv")

print(f"Przed czyszczeniem: {len(df)} rekordów")

# napawia rozbite wiersze (gdy title ma \n + spacje)
df['title'] = df['title'].str.replace(r'\s+\n\s+', ' ', regex=True).str.strip()
df['description'] = df['description'].str.replace(r'\s+\n\s+', ' ', regex=True).str.strip()

# usuwa dokładne duplikaty (gdy wszystkie kolumny identyczne)
df.drop_duplicates(inplace=True)

# usuwa duplikaty po tytule + autorach ### najpierw czyszczenie authors z pustych i spacji
df['authors_clean'] = df['authors'].fillna('').str.replace(r'\s+', ' ', regex=True).str.strip()

df.drop_duplicates(subset=['title', 'authors_clean'], keep='first', inplace=True)
df.drop(columns=['authors_clean'], inplace=True)  ### niepotrzebna po już do usunięcia

# dodatkowe czyszczenie
df['title'] = df['title'].str.strip('"')  # usuwa cudzysłowy z początku/końca
df['publish_year'] = pd.to_numeric(df['publish_year'], errors='coerce')  # poprawia lata

# sortowanie
df.sort_values(['matched_keyword', 'publish_year', 'title'], inplace=True)
df.reset_index(drop=True, inplace=True)

# zapis fixa
output_file = "open_library_articles_fixed.csv"
df.to_csv(output_file, index=False, encoding='utf-8')

print(f"Po czyszczeniu: {len(df)} rekordów (usunięto duplikaty i śmieci)")
print(f"Zapisano do: {output_file}")
print(f"Przykładowe poprawione wiersze:")
print(df[['matched_keyword', 'title', 'authors', 'publish_year']].head(10))

Przed czyszczeniem: 38852 rekordów
Po czyszczeniu: 36851 rekordów (usunięto duplikaty i śmieci)
Zapisano do: open_library_articles_fixed.csv
Przykładowe poprawione wiersze:
  matched_keyword                                              title  \
0         abbasid                  1056 dirham, umaiyadi ed abbasidi   
1         abbasid    A guide to the exhibition of the Abbasid Palace   
2         abbasid                           A short history of Islam   
3         abbasid             A strife between religion and politics   
4         abbasid                                    Abbasid Studies   
5         abbasid       Abbasilere yönelik dinı̂ ve siyası̂ isyanlar   
6         abbasid                                          Abbassaï   
7         abbasid  Abbâsî devleti'nden Hakkâri beyliği'ne İr...   
8         abbasid                                        Abū Muslim   
9         abbasid                  Abū Muslim, sardār-i Khurāsān   

      authors  publish_year  
0   

# update authors

In [2]:
import pandas as pd
import json
from tqdm import tqdm

# wczytanie datasetu
print("Wczytuję open_library_articles_fixed.csv...")
df_books = pd.read_csv("open_library_articles_fixed.csv")
print(f"Rekordów w books: {len(df_books)}")

# przygotowanie listy unikalnych kodów autorów (OLxxxxA) # authors są w formacie: OL123Axxx, OL456Axxx itd.
author_codes = set()
for authors_str in df_books['authors'].dropna():
    for code in authors_str.split(', '):
        code = code.strip()
        if code.startswith('OL') and code.endswith('A'):
            author_codes.add(code)

print(f"Unikalnych kodów autorów do zmapowania: {len(author_codes)}")

# utworzenie mapy kod → nazwisko z authors dump
print("Buduję mapę autorów z ol_dump_authors_2025-11-30.txt...")
author_map = {}

with open("ol_dump_authors_2025-11-30.txt", "r", encoding="utf-8") as f:
    for line_num, line in enumerate(tqdm(f, total=12000000, desc="Przetwarzam authors dump")):
        parts = line.strip().split('\t')
        if len(parts) < 5:
            continue
        json_str = parts[4]
        try:
            data = json.loads(json_str)
            key = data.get('key', '')
            name = data.get('name', '')
            if key and name:
                code = key.split('/')[-1]  # /authors/OL123A → OL123A
                author_map[code] = name
        except:
            continue

print(f"Zmapowano nazwisk autorów: {len(author_map)}")

# mapowanie autorów w books
def map_authors(codes_str):
    if not codes_str or pd.isna(codes_str):
        return ''
    codes = [c.strip() for c in codes_str.split(',')]
    names = []
    for code in codes:
        if code in author_map:
            names.append(author_map[code])
        elif code:  # jeśli nie ma w mapie – zostaw kod
            names.append(code)
    return ', '.join(names)

print("Mapuję autorów w dataset...")
df_books['authors'] = df_books['authors'].apply(map_authors)

# zapis nowego, finalnego pliku
output_final = "open_library_articles_authors_fixed.csv"
df_books.to_csv(output_final, index=False, encoding='utf-8')

print(f"Zapisano {len(df_books)} rekordów z prawdziwymi nazwiskami do: {output_final}")
print("Przykładowe wiersze z autorami:")
print(df_books[['matched_keyword', 'title', 'authors', 'publish_year']].head(15))

Wczytuję open_library_articles_fixed.csv...
Rekordów w books: 36851
Unikalnych kodów autorów do zmapowania: 28431
Buduję mapę autorów z ol_dump_authors_2025-11-30.txt...


Przetwarzam authors dump: 14890320it [00:45, 324059.21it/s]                              

Zmapowano nazwisk autorów: 14888008
Mapuję autorów w dataset...
Zapisano 36851 rekordów z prawdziwymi nazwiskami do: open_library_articles_authors_fixed.csv
Przykładowe wiersze z autorami:
   matched_keyword                                              title  \
0          abbasid                  1056 dirham, umaiyadi ed abbasidi   
1          abbasid    A guide to the exhibition of the Abbasid Palace   
2          abbasid                           A short history of Islam   
3          abbasid             A strife between religion and politics   
4          abbasid                                    Abbasid Studies   
5          abbasid       Abbasilere yönelik dinı̂ ve siyası̂ isyanlar   
6          abbasid                                          Abbassaï   
7          abbasid  Abbâsî devleti'nden Hakkâri beyliği'ne İr...   
8          abbasid                                        Abū Muslim   
9          abbasid                  Abū Muslim, sardār-i Khurāsān   
10      